# 03 - Modèle Baseline (NCF sans contexte)

Ce notebook entraîne un modèle de recommandation de référence sur MovieLens 100K.
Il utilise un NCF simple (`src/models.py`) et traite les interactions comme un problème de feedback binaire.

Objectif : établir un point de référence clair pour mesurer l'impact des features contextuelles dans les notebooks suivants.


In [7]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

sys.path.append(str((Path('..') / 'src').resolve()))
from models import NCF
from metrics import hit_rate, ndcg, mrr
from data_utils import load_movielens_100k, encode_ids

print('Python:', sys.version.split()[0])
print('pandas :', pd.__version__)
print('numpy :', np.__version__)
print('torch :', torch.__version__)


Python: 3.12.13
pandas : 3.0.3
numpy : 2.4.6
torch : 2.12.0+cu130


## Chargement des données

Nous utilisons MovieLens 100K comme dataset de baseline. Si le fichier traité n'existe pas, nous chargeons et encodons directement le dataset brut.


In [8]:
NOTEBOOK_ROOT = Path('.')
RAW_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'raw' / 'movielens' / 'ml-100k').resolve()
PROCESSED_PATH = (NOTEBOOK_ROOT / '..' / 'data' / 'processed' / 'movielens_100k' / 'interactions.parquet').resolve()
print('RAW_ROOT =', RAW_ROOT)
print('PROCESSED_PATH =', PROCESSED_PATH)

if PROCESSED_PATH.exists():
    df = pd.read_parquet(PROCESSED_PATH)
    print('Loaded processed file:', PROCESSED_PATH)
else:
    df = load_movielens_100k(RAW_ROOT)
    df = encode_ids(df)
    print('Loaded raw MovieLens 100K and encoded IDs')

print('Dataset shape:', df.shape)
print(df.columns.tolist())
print(df.head(3).to_string(index=False))


RAW_ROOT = /home/mrtds/Documents/my_projects/context-aware-recsys/data/raw/movielens/ml-100k
PROCESSED_PATH = /home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_100k/interactions.parquet
Loaded processed file: /home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_100k/interactions.parquet
Dataset shape: (99287, 19)
['user_id', 'item_id', 'rating', 'timestamp', 'user_id_encoded', 'item_id_encoded', 'hour_of_day', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'time_since_last_interaction', 'session_id', 'session_length', 'session_position', 'session_position_norm', 'time_since_last_interaction_log']
 user_id  item_id  rating           timestamp  user_id_encoded  item_id_encoded  hour_of_day  day_of_week  is_weekend  hour_sin  hour_cos   dow_sin   dow_cos  time_since_last_interaction  session_id  session_length  session_position  session_position_norm  time_since_last_interaction_log
     259      255     

### Chargement des datasets ml_1m et retailrocket_path

In [9]:
ml_1m_path = '/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_1m/interactions.parquet'
retailrocket_path = '/home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/retailrocket/interactions.parquet'

df_ml_1m = pd.read_parquet(ml_1m_path)
print('====================== Dataset ml-1m ======================')
print('Dataset shape:', df_ml_1m.shape)
print(df_ml_1m.columns.tolist())
print(df_ml_1m.head(3).to_string(index=False))

df_retailrocket = pd.read_parquet(retailrocket_path)
print()
print('====================== Dataset retailrocket ======================')
print('Dataset shape:', df_retailrocket.shape)
print(df_retailrocket.columns.tolist())
print(df_retailrocket.head(3).to_string(index=False))

====================== Dataset ml-1m ======================
Dataset shape: (999611, 19)
['user_id', 'item_id', 'rating', 'timestamp', 'user_id_encoded', 'item_id_encoded', 'hour_of_day', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'time_since_last_interaction', 'session_id', 'session_length', 'session_position', 'session_position_norm', 'time_since_last_interaction_log']
 user_id  item_id  rating           timestamp  user_id_encoded  item_id_encoded  hour_of_day  day_of_week  is_weekend  hour_sin  hour_cos  dow_sin  dow_cos  time_since_last_interaction  session_id  session_length  session_position  session_position_norm  time_since_last_interaction_log
    6040      858       4 2000-04-25 23:05:32             6039              720           23            1           0 -0.258819  0.965926 0.781831  0.62349                          0.0           0              65                 1               0.000000                         0.000000
    6040      593    

## Préparation des labels

Nous convertissons les notes en feedback binaire. Cette approche permet d'entraîner un NCF rapide et comparable avec des modèles implicites.

### 1. Pour le dataset ml-100k

In [10]:
df['label'] = (df['rating'] >= 4).astype(int)
print(df['label'].value_counts(normalize=True))

n_users = df['user_id_encoded'].nunique() if 'user_id_encoded' in df.columns else df['user_id'].nunique()
n_items = df['item_id_encoded'].nunique() if 'item_id_encoded' in df.columns else df['item_id'].nunique()

if 'user_id_encoded' not in df.columns or 'item_id_encoded' not in df.columns:
    df = encode_ids(df)
    n_users = df['user_id_encoded'].nunique()
    n_items = df['item_id_encoded'].nunique()

print('Unique users:', n_users)
print('Unique items:', n_items)
print('Sample interactions:')
print(df[['user_id_encoded', 'item_id_encoded', 'rating', 'label']].head())


label
1    0.555612
0    0.444388
Name: proportion, dtype: float64
Unique users: 943
Unique items: 1349
Sample interactions:
   user_id_encoded  item_id_encoded  rating  label
0              258              253       4      1
1              258              284       4      1
2              258              296       4      1
3              258              183       4      1
4              258              171       4      1


### 2. Pour le dataset ml-1m

In [11]:
df_ml_1m['label'] = (df_ml_1m['rating'] >= 4).astype(int)
print(df_ml_1m['label'].value_counts(normalize=True))

n_users_1m = df_ml_1m['user_id_encoded'].nunique() if 'user_id_encoded' in df_ml_1m.columns else df_ml_1m['user_id'].nunique()
n_items_1m = df_ml_1m['item_id_encoded'].nunique() if 'item_id_encoded' in df_ml_1m.columns else df_ml_1m['item_id'].nunique()

if 'user_id_encoded' not in df_ml_1m.columns or 'item_id_encoded' not in df_ml_1m.columns:
    df_ml_1m = encode_ids(df_ml_1m)
    n_users_1m = df_ml_1m['user_id_encoded'].nunique()
    n_items_1m = df_ml_1m['item_id_encoded'].nunique()

print('Unique users:', n_users_1m)
print('Unique items:', n_items_1m)
print('Sample interactions:')
print(df_ml_1m[['user_id_encoded', 'item_id_encoded', 'rating', 'label']].head())

label
1    0.57529
0    0.42471
Name: proportion, dtype: float64
Unique users: 6040
Unique items: 3416
Sample interactions:
   user_id_encoded  item_id_encoded  rating  label
0             6039              720       4      1
1             6039              549       5      1
2             6039             2020       4      1
3             6039             1630       4      1
4             6039             1688       5      1


### 3. Pour le dataset retailrocket

In [12]:
if 'rating' in df_retailrocket.columns:
    df_retailrocket['label'] = (df_retailrocket['rating'] >= 4).astype(int)
else:
    df_retailrocket['label'] = 1

print(df_retailrocket['label'].value_counts(normalize=True))

if 'user_id_encoded' not in df_retailrocket.columns or 'item_id_encoded' not in df_retailrocket.columns:
    df_retailrocket = encode_ids(df_retailrocket)

n_users_retail = df_retailrocket['user_id_encoded'].nunique()
n_items_retail = df_retailrocket['item_id_encoded'].nunique()

print('Unique users:', n_users_retail)
print('Unique items:', n_items_retail)
print(df_retailrocket[['user_id_encoded', 'item_id_encoded', 'label']].head())


label
1    1.0
Name: proportion, dtype: float64
Unique users: 65968
Unique items: 36126
   user_id_encoded  item_id_encoded  label
0            65586            15263      1
1            46473            12458      1
2             4228            18819      1
3            45960            26184      1
4            65586             3752      1


## Split train/test

Nous utilisons un split simple aléatoire. Pour un benchmark plus solide, on pourra remplacer par un split temporel ultérieurement.


In [13]:
def safe_train_test_split(df, label_col='label'):
    if label_col in df.columns and df[label_col].nunique() > 1:
        stratify = df[label_col]
    else:
        stratify = None
    return train_test_split(df, test_size=0.2, random_state=42, stratify=stratify)

# Pour le ml-100k
train_df, test_df = safe_train_test_split(df)
print('1. Pour le ml-100k')
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)

# Pour le ml-1m
train_df_ml_1m, test_df_ml_1m = safe_train_test_split(df_ml_1m)
print('2. Pour le ml-1m')
print('train shape:', train_df_ml_1m.shape)
print('test shape:', test_df_ml_1m.shape)

# Pour le retailrocket
train_df_retailrocket, test_df_retailrocket = safe_train_test_split(df_retailrocket)
print('3. Pour le retailrocket')
print('train shape:', train_df_retailrocket.shape)
print('test shape:', test_df_retailrocket.shape)

1. Pour le ml-100k
train shape: (79429, 20)
test shape: (19858, 20)
2. Pour le ml-1m
train shape: (799688, 20)
test shape: (199923, 20)
3. Pour le retailrocket
train shape: (625215, 22)
test shape: (156304, 22)


## Dataset PyTorch

Le dataset renvoie les indices utilisateur/item et le label binaire pour l'entraînement.


In [14]:
class MovielensDataset(Dataset):
    def __init__(self, df):
        self.user_ids = torch.tensor(df['user_id_encoded'].values, dtype=torch.long)
        self.item_ids = torch.tensor(df['item_id_encoded'].values, dtype=torch.long)
        self.labels = torch.tensor(df['label'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.item_ids[idx], self.labels[idx]

train_dataset = MovielensDataset(train_df)
test_dataset = MovielensDataset(test_df)

batch_size = 512
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print('Train batches:', len(train_loader))
print('Test batches:', len(test_loader))


Train batches: 156
Test batches: 39


## Modèle baseline

Nous construisons un NCF simple avec `embed_dim=32` et plusieurs couches MLP.


In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = NCF(num_users=n_users, num_items=n_items, embed_dim=32, mlp_layers=[64, 32, 16], dropout=0.2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()

print(model)


Device: cpu
NCF(
  (embedding_user_gmf): Embedding(943, 32)
  (embedding_item_gmf): Embedding(1349, 32)
  (embedding_user_mlp): Embedding(943, 32)
  (embedding_item_mlp): Embedding(1349, 32)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=16, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
  )
  (output_layer): Linear(in_features=48, out_features=1, bias=True)
)


## Entraînement

Boucle d'entraînement classique pour minimiser la perte binaire.


In [16]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for user_batch, item_batch, labels in loader:
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(user_batch, item_batch)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

    return total_loss / len(loader.dataset)

epochs = 8
history = []
for epoch in range(1, epochs + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    history.append(loss)
    print(f'Epoch {epoch}/{epochs} - train loss: {loss:.4f}')


Epoch 1/8 - train loss: 0.6411
Epoch 2/8 - train loss: 0.5698
Epoch 3/8 - train loss: 0.5496
Epoch 4/8 - train loss: 0.5210
Epoch 5/8 - train loss: 0.4860
Epoch 6/8 - train loss: 0.4514
Epoch 7/8 - train loss: 0.4182
Epoch 8/8 - train loss: 0.3847


## Évaluation ranking

Nous évaluons le modèle sur test avec Hit Rate@10, NDCG@10 et MRR.


In [17]:
all_item_ids = torch.arange(n_items, dtype=torch.long, device=device)

def evaluate_ranking(model, df, k=10):
    model.eval()
    hits, ndcgs, mrrs = [], [], []
    with torch.no_grad():
        for _, row in df.iterrows():
            user_id = torch.tensor([row['user_id_encoded']], dtype=torch.long, device=device)
            item_id = int(row['item_id_encoded'])
            user_batch = user_id.expand(n_items)
            logits = model(user_batch, all_item_ids).cpu().numpy()
            ranking = np.argsort(-logits)
            if item_id in ranking[:k]:
                hits.append(1)
            else:
                hits.append(0)
            ndcgs.append(ndcg(ranking, item_id, k))
            mrrs.append(mrr(ranking, item_id))
    return np.mean(hits), np.mean(ndcgs), np.mean(mrrs)

hit10, ndcg10, mrr_score = evaluate_ranking(model, test_df, k=10)
print(f'Hit Rate@10: {hit10:.4f}')
print(f'NDCG@10: {ndcg10:.4f}')
print(f'MRR: {mrr_score:.4f}')


Hit Rate@10: 0.0246
NDCG@10: 0.0116
MRR: 0.0141


## Sauvegarde du modèle baseline

Nous sauvegardons le modèle entraîné pour comparaison future.


In [18]:
RESULTS_DIR = (NOTEBOOK_ROOT / '..' / 'results' / 'models').resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = RESULTS_DIR / 'baseline_ncf_ml100k.pt'
torch.save(model.state_dict(), MODEL_PATH)
print('Saved baseline model to', MODEL_PATH)


Saved baseline model to /home/mrtds/Documents/my_projects/context-aware-recsys/results/models/baseline_ncf_ml100k.pt


## Entraînement et évaluation MovieLens 1M

In [19]:
print('Building MovieLens 1M dataset and model')
train_dataset_ml_1m = MovielensDataset(train_df_ml_1m)
test_dataset_ml_1m = MovielensDataset(test_df_ml_1m)
batch_size_ml_1m = 512
train_loader_ml_1m = DataLoader(train_dataset_ml_1m, batch_size=batch_size_ml_1m, shuffle=True, num_workers=0)
test_loader_ml_1m = DataLoader(test_dataset_ml_1m, batch_size=batch_size_ml_1m, shuffle=False, num_workers=0)
print('Train batches ml-1m:', len(train_loader_ml_1m))
print('Test batches ml-1m:', len(test_loader_ml_1m))
model_ml_1m = NCF(num_users=n_users_1m, num_items=n_items_1m, embed_dim=32, mlp_layers=[64, 32, 16], dropout=0.2).to(device)
optimizer_ml_1m = torch.optim.Adam(model_ml_1m.parameters(), lr=1e-3)
for epoch in range(1, epochs + 1):
    loss = train_epoch(model_ml_1m, train_loader_ml_1m, optimizer_ml_1m, criterion, device)
    print(f'[ml-1m] Epoch {epoch}/{epochs} - train loss: {loss:.4f}')


Building MovieLens 1M dataset and model
Train batches ml-1m: 1562
Test batches ml-1m: 391
[ml-1m] Epoch 1/8 - train loss: 0.5649
[ml-1m] Epoch 2/8 - train loss: 0.5060
[ml-1m] Epoch 3/8 - train loss: 0.4505
[ml-1m] Epoch 4/8 - train loss: 0.4040
[ml-1m] Epoch 5/8 - train loss: 0.3705
[ml-1m] Epoch 6/8 - train loss: 0.3466
[ml-1m] Epoch 7/8 - train loss: 0.3285
[ml-1m] Epoch 8/8 - train loss: 0.3140


In [20]:
print('Evaluating MovieLens 1M model')
def evaluate_ranking_df(model, df, n_items, k=10):
    model.eval()
    hits, ndcgs, mrrs = [], [], []
    all_item_ids = torch.arange(n_items, dtype=torch.long, device=device)
    with torch.no_grad():
        for _, row in df.iterrows():
            user_id = torch.tensor([row['user_id_encoded']], dtype=torch.long, device=device)
            item_id = int(row['item_id_encoded'])
            user_batch = user_id.expand(n_items)
            logits = model(user_batch, all_item_ids).cpu().numpy()
            ranking = np.argsort(-logits)
            hits.append(1 if item_id in ranking[:k] else 0)
            ndcgs.append(ndcg(ranking, item_id, k))
            mrrs.append(mrr(ranking, item_id))
    return np.mean(hits), np.mean(ndcgs), np.mean(mrrs)
hit10_1m, ndcg10_1m, mrr_1m = evaluate_ranking_df(model_ml_1m, test_df_ml_1m, n_items_1m, k=10)
print(f'Hit Rate@10 ml-1m: {hit10_1m:.4f}')
print(f'NDCG@10 ml-1m: {ndcg10_1m:.4f}')
print(f'MRR ml-1m: {mrr_1m:.4f}')


Evaluating MovieLens 1M model
Hit Rate@10 ml-1m: 0.0141
NDCG@10 ml-1m: 0.0067
MRR ml-1m: 0.0079


## Entraînement et évaluation RetailRocket

In [21]:
print('Building RetailRocket dataset and model')
train_dataset_retailrocket = MovielensDataset(train_df_retailrocket)
test_dataset_retailrocket = MovielensDataset(test_df_retailrocket)
batch_size_retailrocket = 512
train_loader_retailrocket = DataLoader(train_dataset_retailrocket, batch_size=batch_size_retailrocket, shuffle=True, num_workers=0)
test_loader_retailrocket = DataLoader(test_dataset_retailrocket, batch_size=batch_size_retailrocket, shuffle=False, num_workers=0)
print('Train batches retailrocket:', len(train_loader_retailrocket))
print('Test batches retailrocket:', len(test_loader_retailrocket))
model_retailrocket = NCF(num_users=n_users_retail, num_items=n_items_retail, embed_dim=32, mlp_layers=[64, 32, 16], dropout=0.2).to(device)
optimizer_retailrocket = torch.optim.Adam(model_retailrocket.parameters(), lr=1e-3)
for epoch in range(1, epochs + 1):
    loss = train_epoch(model_retailrocket, train_loader_retailrocket, optimizer_retailrocket, criterion, device)
    print(f'[retailrocket] Epoch {epoch}/{epochs} - train loss: {loss:.4f}')


Building RetailRocket dataset and model
Train batches retailrocket: 1222
Test batches retailrocket: 306
[retailrocket] Epoch 1/8 - train loss: 0.0370
[retailrocket] Epoch 2/8 - train loss: 0.0000
[retailrocket] Epoch 3/8 - train loss: 0.0000
[retailrocket] Epoch 4/8 - train loss: 0.0000
[retailrocket] Epoch 5/8 - train loss: 0.0000
[retailrocket] Epoch 6/8 - train loss: 0.0000
[retailrocket] Epoch 7/8 - train loss: 0.0000
[retailrocket] Epoch 8/8 - train loss: 0.0000


In [22]:
print('Evaluating RetailRocket model')
hit10_retail, ndcg10_retail, mrr_retail = evaluate_ranking_df(model_retailrocket, test_df_retailrocket, n_items_retail, k=10)
print(f'Hit Rate@10 retailrocket: {hit10_retail:.4f}')
print(f'NDCG@10 retailrocket: {ndcg10_retail:.4f}')
print(f'MRR retailrocket: {mrr_retail:.4f}')


Evaluating RetailRocket model
Hit Rate@10 retailrocket: 0.0074
NDCG@10 retailrocket: 0.0044
MRR retailrocket: 0.0048


### Sauvegarde des models ml-1m et retailrocket

In [23]:
# Sauvegarder le modèle ML-1M
MODEL_PATH_ML_1M = RESULTS_DIR / 'baseline_ncf_ml1m.pt'
torch.save(model_ml_1m.state_dict(), MODEL_PATH_ML_1M)
print('Saved ML-1M model to', MODEL_PATH_ML_1M)

# Sauvegarder le modèle RetailRocket
MODEL_PATH_RETAILROCKET = RESULTS_DIR / 'baseline_ncf_retailrocket.pt'
torch.save(model_retailrocket.state_dict(), MODEL_PATH_RETAILROCKET)
print('Saved RetailRocket model to', MODEL_PATH_RETAILROCKET)

Saved ML-1M model to /home/mrtds/Documents/my_projects/context-aware-recsys/results/models/baseline_ncf_ml1m.pt
Saved RetailRocket model to /home/mrtds/Documents/my_projects/context-aware-recsys/results/models/baseline_ncf_retailrocket.pt


## Résumé

Ce notebook entraîne et évalue des modèles baseline NCF sur trois datasets:
- **MovieLens 100K**: 943 utilisateurs, 1349 items
- **MovieLens 1M**: ~6K utilisateurs, ~4K items  
- **RetailRocket**: E-commerce avec interactions implicites

Pour chaque dataset, nous avons:
1. Préparé les labels (feedback binaire: rating ≥ 4)
2. Effectué un split train/test 80/20
3. Entraîné un modèle NCF avec embed_dim=32 et MLPs [64, 32, 16]
4. Évalué avec Hit Rate@10, NDCG@10 et MRR
5. Sauvegardé les modèles pour comparaison

La comparaison de ces métriques baseline avec des modèles contextuels permettra de quantifier le gain apporté par les features temporelles et de session.